In [1]:
from pathlib import Path 
import os
import pandas as pd
import numpy as np
import dotenv
dotenv.load_dotenv()
os.chdir(Path(os.getenv("PYTHONPATH")).expanduser())

In [3]:
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader

from src.util import build_windows, split_indices
from src.data import LorenzWindowViewDataset

# --- load and pick channels ---
df = pd.read_pickle("data/lorenz_data.pkl")
df = df.iloc[ : 1000 ]   # shorten for testing
series = df[["x", "y", "z"]].to_numpy(dtype=np.float32)      # (T, 3)
mean   = series.mean(axis=0).astype(np.float32)
std    = series.std(axis=0).astype(np.float32)
std    = np.where(std == 0, 1.0, std)

# --- make a window VIEW ---
WARM, XLEN = 10, 100
windows = build_windows(series, warm_length=WARM, x_length=XLEN)   # (N, W+X+1, 3) view
N = windows.shape[0]

# --- split into *indices* (not materialized arrays) ---
train_idx, test_idx = split_indices(N, train_ratio=0.8, seed=42)

# (optional) sample down to a manageable number of windows
MAX_WINS = 500_000
if N > MAX_WINS:
    rng = np.random.default_rng(42)
    keep = rng.choice(N, size=MAX_WINS, replace=False)
    tr_keep, te_keep = split_indices(MAX_WINS, train_ratio=0.8, seed=43)
    train_idx, test_idx = keep[tr_keep], keep[te_keep]

# --- datasets & loaders (zero-copy from view) ---
train_ds = LorenzWindowViewDataset(windows, train_idx, WARM, XLEN, standardize=True, mean=mean, std=std)
test_ds  = LorenzWindowViewDataset(windows, test_idx,  WARM, XLEN, standardize=True, mean=mean, std=std)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=4, shuffle=False, num_workers=0, pin_memory=False)

# sanity check
wb, xb, yb = next(iter(train_loader))
print(wb.shape, xb.shape, yb.shape)   # (B, W, 3) (B, X, 3) (B, X, 3)


torch.Size([4, 10, 3]) torch.Size([4, 100, 3]) torch.Size([4, 100, 3])


In [4]:
print(f"train_loader has {len(train_loader.dataset)} windows")
print(f"test_loader has {len(test_loader.dataset)} windows")

train_loader has 712 windows
test_loader has 178 windows


In [5]:
import torch
import torch.nn as nn
from src.model.ESN import ESNRegressor

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available() and False:  # MPS currently has issues with some operations
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

model = ESNRegressor(
    input_size=3, output_size=3,
    reservoir_size=500, leak=0.7,
    spectral_radius=0.9, input_scale=0.8,
    sparsity=0.05, bias_scale=0.0, activation="tanh", seed=42,
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

def train_epoch(loader):
    model.train()
    running = 0.0
    for warm, x, y in loader:
        warm = warm.to(device, non_blocking=True)
        x    = x.to(device, non_blocking=True)
        y    = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        y_hat = model.forward_seq(warm, x)   # (B, X, D)
        loss = criterion(y_hat, y)
        loss.backward()
        # optional: gradient clip
        # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running += loss.item()
    return running / max(1, len(loader))

@torch.no_grad()
def eval_epoch(loader):
    model.eval()
    running = 0.0
    for warm, x, y in loader:
        warm = warm.to(device, non_blocking=True)
        x    = x.to(device, non_blocking=True)
        y    = y.to(device, non_blocking=True)
        y_hat = model.forward_seq(warm, x)
        running += criterion(y_hat, y).item()
    return running / max(1, len(loader))

EPOCHS = 3
for epoch in range(1, EPOCHS+1):
    tr = train_epoch(train_loader)
    te = eval_epoch(test_loader)
    print(f"Epoch {epoch}: train {tr:.6f} | test {te:.6f}")


Using device: cpu
Epoch 1: train 0.039820 | test 0.001744
Epoch 2: train 0.001071 | test 0.000498
Epoch 3: train 0.000470 | test 0.000381
